# BetterTogether (CUDA)

`BetterTogether` alternates prompt optimization (`p`) and weight optimization
(`w`). The keyword names passed to the constructor define the strategy symbols;
this notebook therefore uses `p=...`, `w=...` and `strategy="p -> w -> p"`.
It requires the same CUDA/SGLang stack as `BootstrapFinetune`.

In [ ]:
%pip install -r ../requirements.txt -q torch transformers datasets accelerate peft trl "sglang[all]"

In [ ]:
import os
import random
from pathlib import Path

import dspy
import pandas as pd
from dotenv import load_dotenv

# Resolve paths whether Jupyter starts in the repo root or chapter06/.
cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "data" / "ai_vs_human200.csv").exists() else cwd.parent
DATA_PATH = REPO_ROOT / "data" / "ai_vs_human200.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("Start Jupyter in the repo root or chapter06 directory.")

load_dotenv(REPO_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("Set OPENAI_API_KEY in the repo's .env file before running this notebook.")

# Chapter 6 defaults: Luna for high-volume task calls, Sol for reflection.
# Override either slug without editing the notebook.
TASK_MODEL = os.getenv("TASK_MODEL", "openai/gpt-5.6-luna")
REFLECTION_MODEL = os.getenv("REFLECTION_MODEL", "openai/gpt-5.6-sol")
NUM_THREADS = int(os.getenv("DSPY_NUM_THREADS", "4"))
TRAIN_LIMIT = int(os.getenv("TRAIN_LIMIT", "32"))
VAL_LIMIT = int(os.getenv("VAL_LIMIT", "12"))
EVAL_LIMIT = int(os.getenv("EVAL_LIMIT", "12"))

# DSPy 3.2 treats GPT-5-family models as reasoning models and rejects small
# max_tokens caps. Leave the provider default in place rather than setting an
# invalid value below 16,000.
task_lm = dspy.LM(TASK_MODEL)
reflection_lm = dspy.LM(REFLECTION_MODEL)
dspy.configure(lm=task_lm)

print(f"DSPy {dspy.__version__}")
print(f"Task model: {TASK_MODEL}")
print(f"Reflection model: {REFLECTION_MODEL}")

In [ ]:
class DetectAIText(dspy.Signature):
    """Decide whether the supplied passage was generated by an AI."""

    text: str = dspy.InputField(desc="Passage to classify")
    is_ai: bool = dspy.OutputField(desc="True for AI-generated text; otherwise False")


class AIDetector(dspy.Module):
    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(DetectAIText)

    def forward(self, text: str):
        return self.detect(text=text)


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    normalized = str(value).strip().lower()
    if normalized in {"true", "1", "yes"}:
        return True
    if normalized in {"false", "0", "no"}:
        return False
    raise ValueError(f"Cannot interpret {value!r} as a boolean")


def exact_match(example, prediction, trace=None) -> float:
    return float(as_bool(prediction.is_ai) == bool(example.is_ai))


def feedback_metric(example, prediction, trace=None, **kwargs):
    score = exact_match(example, prediction)
    feedback = (
        "The classification is correct. Preserve the reasoning cues that led to it."
        if score
        else f"The classification is incorrect. The expected is_ai value is {bool(example.is_ai)}. "
             "Focus on concrete stylistic evidence instead of topic or polish alone."
    )
    return dspy.Prediction(score=score, feedback=feedback)


frame = pd.read_csv(DATA_PATH)
examples = [
    dspy.Example(text=row.text, is_ai=bool(row.is_ai)).with_inputs("text")
    for row in frame.itertuples(index=False)
]
random.Random(42).shuffle(examples)

# Deterministic 50/25/25 split; environment variables keep default runs inexpensive.
train_end = len(examples) // 2
val_end = train_end + len(examples) // 4
full_trainset, full_valset, full_testset = (
    examples[:train_end], examples[train_end:val_end], examples[val_end:]
)
trainset = full_trainset[:TRAIN_LIMIT] if TRAIN_LIMIT else full_trainset
valset = full_valset[:VAL_LIMIT] if VAL_LIMIT else full_valset
testset = full_testset[:EVAL_LIMIT] if EVAL_LIMIT else full_testset

def evaluate(program, dataset=testset):
    runner = dspy.Evaluate(
        devset=dataset,
        metric=exact_match,
        num_threads=NUM_THREADS,
        max_errors=max(1, len(dataset)),
        provide_traceback=True,
        failure_score=0.0,
        display_progress=True,
        display_table=5,
    )
    result = runner(program)
    failed = [index for index, (_, prediction, _) in enumerate(result.results) if not hasattr(prediction, "is_ai")]
    if failed:
        raise RuntimeError(f"Evaluation failed for example indexes {failed}; inspect the tracebacks above.")
    return result.score


detector = AIDetector()
print(f"train={len(trainset)}, validation={len(valset)}, test={len(testset)}")

In [ ]:
import torch
from dspy.clients.lm_local import LocalProvider

if not torch.cuda.is_available():
    raise RuntimeError("BetterTogether weight optimization requires an NVIDIA CUDA GPU.")

LOCAL_MODEL = os.getenv("LOCAL_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
local_lm = dspy.LM(
    f"openai/local:{LOCAL_MODEL}",
    provider=LocalProvider(),
    max_tokens=512,
)
student = AIDetector()
student.set_lm(local_lm)

prompt_optimizer = dspy.MIPROv2(
    metric=exact_match,
    prompt_model=reflection_lm,
    task_model=local_lm,
    auto="light",
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
    num_threads=NUM_THREADS,
)
weight_optimizer = dspy.BootstrapFinetune(
    metric=exact_match,
    multitask=False,
    train_kwargs={
        "num_train_epochs": 1,
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "max_seq_length": 1_024,
        "device": "cuda",
        "use_peft": True,
    },
)

optimizer = dspy.BetterTogether(
    metric=exact_match,
    p=prompt_optimizer,
    w=weight_optimizer,
)
optimized_detector = optimizer.compile(
    student,
    trainset=trainset,
    valset=valset,
    strategy="p -> w -> p",
)

In [ ]:
# BetterTogether cleans up local servers before returning.
optimized_lms = {predictor.lm for predictor in optimized_detector.predictors()}
for lm in optimized_lms:
    lm.launch()
try:
    optimized_score = evaluate(optimized_detector)
    print(f"BetterTogether score: {optimized_score}")
finally:
    for lm in optimized_lms:
        lm.kill()